# Traditional ML — Interview Q&A

> **Target roles:** Research Scientist · ML Engineer · Foundational Modeling · Traditional ML Modeling  
> Covers: Supervised/Unsupervised learning, SVM, regularization, bias-variance tradeoff, feature engineering, evaluation metrics, and more.  
> ⚠️ marks trick questions.

## 1. Fundamentals: Bias-Variance & Model Selection

**Q1. Explain the bias-variance tradeoff.**  
**Bias:** Error from wrong assumptions in the learning algorithm (underfitting). High-bias models are too simple.  
**Variance:** Error from sensitivity to fluctuations in the training set (overfitting). High-variance models memorize noise.  
Total Error = Bias² + Variance + Irreducible Noise. The goal is to find the sweet spot. Increasing model complexity decreases bias but increases variance. Regularization, ensembles, and more data can help manage this tradeoff.

**Q2. How do you detect underfitting vs overfitting?**  
**Underfitting:** High training error AND high validation error. Model too simple for the data.  
**Overfitting:** Low training error but high validation error — large train/val gap. Solutions: more data, regularization, simpler model, dropout, early stopping.  
Plot learning curves (train/val error vs dataset size) to diagnose: converging curves = underfitting; large gap = overfitting.

**Q3. Explain cross-validation. When would you use K-Fold vs Stratified K-Fold vs TimeSeriesSplit?**  
**K-Fold CV:** Splits data into K folds, trains on K-1, validates on 1, rotates. Reduces variance of metric estimate.  
**Stratified K-Fold:** Preserves class distribution in each fold. Always use for imbalanced classification.  
**TimeSeriesSplit:** Respects temporal ordering — train on past, validate on future. Critical for time-series to prevent data leakage.  
**LOOCV:** K=N. Very low bias but high variance and expensive. Use for small datasets (<100 samples).

**Q4. What is the curse of dimensionality?**  
In high dimensions: (1) data becomes sparse — all points are far from each other; (2) distance metrics lose meaning (ratio of max/min distance → 1); (3) volume grows exponentially — need exponentially more data to maintain sample density; (4) models require exponentially more samples to generalize. Mitigations: PCA/dimensionality reduction, feature selection, regularization, manifold learning.

**Q5. Explain the no free lunch theorem.**  
No single algorithm is best for all problems. For every problem where algorithm A outperforms B, there exists another problem where B outperforms A (averaged over all possible problems). Practically: always try multiple algorithms, understand your data's structure, and choose based on empirical validation — not faith in one method.


## 2. Linear Models & Regularization

**Q6. What is the difference between L1 (Lasso) and L2 (Ridge) regularization?**  
**L2 (Ridge):** Loss + λΣwᵢ². Gradient ∝ wᵢ — weights decay smoothly to small values, never exactly zero. Prefers many small weights. Closed-form solution exists: W = (XᵀX + λI)⁻¹XᵀY. Good when all features are relevant.  
**L1 (Lasso):** Loss + λΣ|wᵢ|. Subgradient is ±λ regardless of weight magnitude — forces some weights to *exactly zero*. Produces sparse solutions; acts as feature selection. No closed-form solution. Good when few features are relevant.  
**Elastic Net:** L1 + L2 combined. Gets sparsity of Lasso + grouping effect of Ridge. Best for correlated features.

**Q7. Why does L1 produce sparse solutions but L2 doesn't?**  
Geometric interpretation: L1 constraint is a diamond (sharp corners on axes) — the optimum tends to hit a corner (sparse). L2 constraint is a sphere (smooth, no corners) — the optimum rarely hits an axis. Probability interpretation: L1 ↔ Laplace prior (heavy tails favor zeros), L2 ↔ Gaussian prior.

**Q8. What is logistic regression and how is it trained?**  
Linear model for binary classification: P(y=1|x) = σ(wᵀx + b) where σ is sigmoid. Trained by maximizing log-likelihood (equivalent to minimizing binary cross-entropy). No closed-form solution — use gradient descent or Newton's method (IRLS). Despite the name, it's a *classification* algorithm. Outputs calibrated probabilities.

**Q9. What is multicollinearity and how does it affect models?**  
When features are highly correlated, the design matrix XᵀX becomes near-singular — OLS estimates are unstable with huge variance. Small changes in data cause large changes in coefficients. Solutions: Ridge regression (adds λI to make XᵀX + λI invertible), PCA, VIF-based feature removal. L1 (Lasso) also helps by zeroing out redundant features.

**Q10. Explain gradient descent variants: batch, mini-batch, stochastic.**  
**Batch GD:** Uses all N samples per update. Stable gradient, slow per iteration, memory-intensive.  
**SGD:** Uses 1 sample per update. Noisy gradient, fast, can escape local minima, but high variance.  
**Mini-batch GD:** Uses B samples (typically 32–512). Best of both: vectorized, moderate noise, works well with modern hardware. Almost universally used in practice.


## 3. Support Vector Machines (SVM)

**Q11. Explain SVM — what is it maximizing?**  
SVM finds the hyperplane wᵀx + b = 0 that maximizes the **margin** — the distance between the hyperplane and the nearest points (support vectors) of each class. Margin = 2/‖w‖. So maximizing margin = minimizing ‖w‖²/2 subject to yᵢ(wᵀxᵢ + b) ≥ 1. This is a constrained quadratic optimization problem solved via dual form with Lagrange multipliers.

**Q12. What is the kernel trick?**  
SVMs find a linear separator in the feature space. For non-linearly separable data, we implicitly map inputs to a high (possibly infinite) dimensional space using a kernel function K(x,x') = φ(x)ᵀφ(x') without computing φ explicitly. Common kernels: Linear, Polynomial K=(xᵀx'+c)^d, RBF/Gaussian K=exp(-γ‖x-x'‖²), Sigmoid. The dual SVM formulation only needs dot products, so kernels are efficient.

**Q13. What is the C parameter in SVM?**  
C is the regularization parameter (inverse of regularization strength). **Small C:** Wider margin, more misclassifications allowed (soft margin) — higher bias, lower variance. **Large C:** Narrow margin, penalizes misclassification heavily (hard margin) — lower bias, higher variance. Tune via cross-validation.

**Q14. What is the γ (gamma) parameter in RBF kernel SVM?**  
γ = 1/(2σ²) controls the width of the Gaussian. **Small γ:** Wide kernel, each point influences a large region → smoother decision boundary, underfitting risk. **Large γ:** Narrow kernel, points only influence nearby region → complex decision boundary, overfitting risk. γ and C must be tuned together (grid search in log scale).

**Q15. When would you choose SVM over logistic regression or tree-based models?**  
SVM advantages: works well in high-dimensional spaces (text classification), effective when #features > #samples, memory-efficient (only support vectors matter), kernel trick for non-linear boundaries. Disadvantages: doesn't scale well to large datasets (O(N²-N³) training), no probability calibration by default, sensitive to feature scaling, hard to interpret. Use tree-based models for large datasets; SVM for small, high-dimensional problems.


## 4. Unsupervised Learning

**Q16. Explain K-Means clustering. What are its limitations?**  
Algorithm: (1) Initialize K centroids randomly; (2) Assign each point to nearest centroid; (3) Update centroids to cluster means; (4) Repeat until convergence. Minimizes within-cluster sum of squares (WCSS).  
**Limitations:** K must be specified (use elbow method or silhouette score); assumes spherical, equally-sized clusters; sensitive to initialization (use K-Means++); sensitive to outliers; not suitable for non-convex clusters.

**Q17. How do you choose K in K-Means?**  
**Elbow method:** Plot WCSS vs K; find the "elbow" where adding more clusters gives diminishing returns.  
**Silhouette score:** Measures how similar a point is to its own cluster vs other clusters. Range [-1, 1]; higher is better.  
**Gap statistic:** Compares WCSS to a reference null distribution. Most principled but expensive.  
**Domain knowledge:** Often the best guide.

**Q18. Explain PCA — what does it do geometrically?**  
PCA finds the orthogonal directions (principal components) of maximum variance in the data. Geometrically: rotates the coordinate system to align with the directions of greatest spread. First PC = direction of maximum variance; subsequent PCs = orthogonal, decreasing variance. Implemented via SVD of the centered data matrix X = UΣVᵀ. PCs are the columns of V; projections are XV.

**Q19. What is the difference between PCA and t-SNE / UMAP?**  
**PCA:** Linear, preserves global structure (variance), fast, deterministic, can project new points. Use for dimensionality reduction before modeling.  
**t-SNE:** Non-linear, preserves local structure (neighbors), not deterministic, cannot project new points, sensitive to perplexity. Use for visualization only.  
**UMAP:** Non-linear, preserves both local and some global structure, faster than t-SNE, can project new points. Better default choice for visualization.

**Q20. What is DBSCAN and how does it differ from K-Means?**  
DBSCAN (Density-Based Spatial Clustering): groups points that are densely packed (within ε of each other), with at least minPts neighbors. Key differences: (1) doesn't require K; (2) finds arbitrarily shaped clusters; (3) identifies outliers as noise; (4) struggles with varying density clusters. Parameters: ε (neighborhood radius) and minPts (minimum cluster size).


## 5. Feature Engineering & Data Preprocessing

**Q21. What are the main feature encoding techniques for categorical variables?**  
**One-hot encoding:** Creates binary columns. Good for low-cardinality nominals. Problem: dimensionality explosion for high cardinality.  
**Label encoding:** Assigns integers. Only valid for ordinal variables (introduces false ordering otherwise).  
**Target encoding:** Replace category with mean of target. Powerful but requires cross-fitting to avoid leakage.  
**Embedding:** Learn dense representations (for neural nets or high-cardinality features).  
**Frequency/count encoding:** Replace with category frequency. Simple, avoids high dimensionality.

**Q22. What is feature scaling and when is it needed?**  
**Standardization (Z-score):** (x-μ)/σ. Makes features zero-mean, unit-variance. Needed by: SVM, KNN, logistic regression, PCA, neural networks.  
**Min-Max normalization:** (x-min)/(max-min) → [0,1]. Sensitive to outliers.  
**Not needed by:** Tree-based models (Random Forest, XGBoost, decision trees) — they use splits, not distances.  
**Robust scaling:** Uses median and IQR — insensitive to outliers. Use when data has many outliers.

**Q23. How do you handle missing values?**  
**Options:** (1) Drop rows/columns (only if <5% missing and random); (2) Mean/median/mode imputation; (3) Model-based imputation (KNN, MICE/iterative imputation); (4) Indicator variable (create a "was_missing" flag — often informative); (5) Some models handle natively (XGBoost, LightGBM).  
Critical: Fit imputer on *training data only*, then transform both train and test to prevent data leakage.

**Q24. What is target leakage and how do you prevent it?**  
Target leakage: using information at prediction time that wouldn't be available in production (e.g., features computed using the target, or future data). Causes model to appear excellent in validation but fail in production.  
Prevention: (1) Carefully track feature timestamps vs target timestamp; (2) Use time-aware splits for time-series; (3) Fit preprocessors (scalers, encoders, imputers) only on training fold; (4) Watch for features highly correlated with target.


## 6. Evaluation Metrics

**Q25. Explain precision, recall, F1 score.**  
**Precision:** TP/(TP+FP) — of predicted positives, how many are actually positive? Optimize when false positives are costly (e.g., spam filter).  
**Recall (Sensitivity):** TP/(TP+FN) — of actual positives, how many did we catch? Optimize when false negatives are costly (e.g., cancer detection).  
**F1:** Harmonic mean of precision and recall = 2PR/(P+R). Balances both. Use when class imbalance exists.  
**F-beta:** ((1+β²)·P·R) / (β²·P + R). β>1 emphasizes recall; β<1 emphasizes precision.

**Q26. What is ROC-AUC and when is it appropriate?**  
ROC curve: plots TPR (recall) vs FPR (fall-out) across all classification thresholds. AUC = probability that the model ranks a random positive higher than a random negative (C-statistic). AUC=0.5 is random; AUC=1 is perfect.  
**When to use:** Binary classification, comparing models regardless of threshold, imbalanced classes (more robust than accuracy). **Limitation:** Can be misleading when class imbalance is extreme — use Precision-Recall AUC instead.

**Q27. What metrics do you use for regression?**  
**MAE:** Mean absolute error. Robust to outliers, same unit as target.  
**MSE/RMSE:** Mean squared/root-mean-squared error. Penalizes large errors heavily. RMSE in same unit as target.  
**MAPE:** Mean absolute percentage error. Scale-independent but undefined for zero targets.  
**R²:** Fraction of variance explained (1 - SS_res/SS_tot). Scale-independent, interpretable. Can be negative for terrible models.  
Choose based on domain: RMSE when large errors are very costly; MAE for robust summary; MAPE for comparing across scales.

**Q28. How do you handle class imbalance?**  
**Data-level:** Oversample minority (SMOTE creates synthetic samples), undersample majority, class weights.  
**Algorithm-level:** `class_weight='balanced'` in sklearn, `scale_pos_weight` in XGBoost.  
**Metric-level:** Use F1, PR-AUC, MCC (Matthews Correlation Coefficient) instead of accuracy.  
**Threshold adjustment:** Move classification threshold from 0.5 to optimize for your metric.  
Key mistake: Never oversample before cross-validation splits — always oversample inside each fold to prevent leakage.


## 7. Decision Trees

**Q29. How does a decision tree make splits?**  
At each node, exhaustively searches all features and all split points to maximize the information gain (or Gini gain). **Information gain:** IG = H(parent) - Σ(|child|/|parent|)·H(child) where H is entropy. **Gini:** G = 1 - Σpᵢ². Both measure impurity reduction. The feature+threshold with the highest gain is chosen.

**Q30. What are the hyperparameters of a decision tree and how do they affect bias-variance?**  
`max_depth`: Limits tree depth. Low → high bias (underfit). High → high variance (overfit).  
`min_samples_split` / `min_samples_leaf`: Minimum samples to split/in a leaf. Higher → simpler tree, more bias, less variance.  
`max_features`: Number of features considered at each split (key to Random Forest).  
`ccp_alpha`: Cost-complexity pruning. Higher = more pruning = simpler tree.


## 8. ⚠️ Trick Questions

**Q31. ⚠️ Is accuracy a good metric for imbalanced datasets?**  
No! A model that always predicts the majority class achieves high accuracy trivially. Example: 99% negative class → 99% accuracy by always predicting negative. Use F1, PR-AUC, or MCC instead. AUC-ROC is also better than accuracy but can still be misleading with extreme imbalance.

**Q32. ⚠️ Does feature scaling affect Random Forest or XGBoost?**  
No. Tree-based models split on thresholds, so monotone transformations (scaling, log) don't change the split decisions. Feature scaling only matters for distance-based (KNN, SVM) and gradient-based (logistic regression, neural networks, PCA) algorithms.

**Q33. ⚠️ Why shouldn't you use K-Fold CV for time series data?**  
Standard K-Fold randomly shuffles data, so training folds can include future data relative to the validation fold — this is data leakage. In production, you'll only have past data to predict the future. Use TimeSeriesSplit (walk-forward validation) instead, where train always precedes validation chronologically.

**Q34. ⚠️ Is a higher R² always better?**  
Not necessarily. R² can be artificially inflated by adding useless features (always increases or stays the same with more features). Use Adjusted R² = 1 - (1-R²)(N-1)/(N-p-1) which penalizes extra features. Also, high R² doesn't mean good predictions in absolute terms, nor does it validate the model's assumptions.

**Q35. ⚠️ Can logistic regression output probabilities that are well-calibrated?**  
Logistic regression is generally well-calibrated for its training distribution, but calibration degrades with regularization, data imbalance, or distribution shift. Random Forest and SVM are typically poorly calibrated. Use Platt scaling or isotonic regression to post-hoc calibrate any classifier.

**Q36. ⚠️ What's wrong with using the test set for hyperparameter tuning?**  
It causes test set overfitting — the model is indirectly trained on the test set through the tuning process. The test set should be touched exactly once for the final evaluation. Use a held-out validation set or cross-validation for tuning.

**Q37. ⚠️ Does more data always help?**  
Not always. More data helps reduce variance. But if: (1) the model is fundamentally misspecified (high bias), more data barely helps; (2) the additional data has different distribution (covariate shift); (3) there is significant label noise. Also: for low-variance models like linear regression on simple problems, there's a saturation point.


## 9. Advanced Topics

**Q38. What is the EM (Expectation-Maximization) algorithm?**  
Iterative algorithm for maximum likelihood estimation with latent variables. **E-step:** Compute expected value of the complete-data log-likelihood given current parameters. **M-step:** Maximize expected log-likelihood to get new parameters. Repeat until convergence. Applications: Gaussian Mixture Models (GMMs), HMMs. Guaranteed to increase likelihood each iteration but may converge to local maximum.

**Q39. What is a Gaussian Mixture Model?**  
Probabilistic model assuming data is generated from K Gaussian distributions with unknown means, covariances, and mixing weights. Trained with EM. Soft clustering: each point has a probability of belonging to each cluster. More flexible than K-Means (non-spherical clusters, soft assignments). Choose K via BIC/AIC (penalized likelihood).

**Q40. What is the difference between generative and discriminative models?**  
**Generative:** Models joint distribution P(X,Y) = P(X|Y)·P(Y). Can generate new samples. Examples: Naive Bayes, GMMs, VAEs, GANs. More parameters, can work with less labeled data.  
**Discriminative:** Models P(Y|X) directly. Examples: logistic regression, SVM, neural networks. Usually better classification performance given enough data (learns the decision boundary, not the full data distribution).

**Q41. Explain Naive Bayes and when it works despite its naive assumption.**  
Naive Bayes applies Bayes' theorem assuming features are *conditionally independent* given the class: P(y|x₁,...,xₙ) ∝ P(y)·Πᵢ P(xᵢ|y). The naive assumption is almost always violated in practice, yet NB often works well because: (1) classification only requires the correct *ranking* of class posteriors, not accurate probabilities; (2) errors in feature probabilities can cancel; (3) works very well for text classification (bag-of-words).


## 10. Quick Reference Summary

### Algorithm Selection Guide
| Scenario | Start With |
|---|---|
| Tabular data, medium size | XGBoost / LightGBM |
| High-dimensional sparse (text) | Logistic Regression / LinearSVC |
| Small dataset, non-linear | SVM with RBF kernel |
| Interpretability required | Decision Tree / Logistic Regression |
| Clustering | K-Means (spherical) / DBSCAN (arbitrary shape) |
| Anomaly detection | Isolation Forest / One-Class SVM |
| Dimensionality reduction | PCA (linear) / UMAP (visualization) |

### Regularization Cheatsheet
| Regularization | Effect | Best When |
|---|---|---|
| L1 (Lasso) | Sparse weights, feature selection | Few relevant features |
| L2 (Ridge) | Small weights, no zeros | All features potentially relevant |
| Elastic Net | Sparse + grouped | Correlated features |
| Dropout | Random masking of neurons | Deep networks |
| Early stopping | Implicit complexity limit | Any model with iterative training |

### Metrics Cheatsheet
| Problem | Metric |
|---|---|
| Balanced classification | Accuracy, F1 |
| Imbalanced classification | F1, PR-AUC, MCC |
| Ranking/scoring | ROC-AUC |
| Regression (outliers) | MAE, Huber loss |
| Regression (large errors costly) | RMSE |
| Probability calibration | Brier score, ECE |

### Common Interview Mistakes
- Using accuracy for imbalanced data  
- Fitting preprocessors on the full dataset (leakage!)  
- Forgetting to scale features for SVM/KNN  
- Using standard K-Fold for time-series  
- Confusing L1/L2 effects on weights  
- Saying "more data always helps"  


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_regression

# Demonstrate L1 vs L2 sparsity
np.random.seed(42)
X, y, true_coef = make_regression(n_samples=100, n_features=20, n_informative=5,
                                   coef=True, noise=10, random_state=42)
X = StandardScaler().fit_transform(X)

alphas = np.logspace(-3, 3, 50)
ridge_coefs, lasso_coefs = [], []
for a in alphas:
    ridge_coefs.append(Ridge(alpha=a).fit(X, y).coef_)
    lasso_coefs.append(Lasso(alpha=a, max_iter=5000).fit(X, y).coef_)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for coef_path in np.array(ridge_coefs).T:
    ax1.plot(np.log10(alphas), coef_path, alpha=0.7)
ax1.set_xlabel('log10(alpha)'); ax1.set_ylabel('Coefficient value')
ax1.set_title('Ridge (L2): Coefficients shrink smoothly, never zero')
ax1.axhline(0, color='k', lw=0.5); ax1.grid(True, alpha=0.3)

for coef_path in np.array(lasso_coefs).T:
    ax2.plot(np.log10(alphas), coef_path, alpha=0.7)
ax2.set_xlabel('log10(alpha)'); ax2.set_ylabel('Coefficient value')
ax2.set_title('Lasso (L1): Coefficients hit exactly zero (sparsity!)')
ax2.axhline(0, color='k', lw=1.5, ls='--'); ax2.grid(True, alpha=0.3)

plt.suptitle('L1 vs L2 Regularization: Coefficient Paths', fontsize=13)
plt.tight_layout()
plt.savefig('l1_vs_l2.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"True non-zero coefficients: {(true_coef != 0).sum()}")


In [ ]:
# Bias-Variance decomposition visualization
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline

np.random.seed(42)
def true_function(x): return np.sin(2 * np.pi * x)

n_experiments = 100
x_test = np.linspace(0, 1, 100)
predictions = {depth: [] for depth in [1, 3, 10]}

for _ in range(n_experiments):
    x_train = np.random.uniform(0, 1, 30)
    y_train = true_function(x_train) + np.random.normal(0, 0.3, 30)
    for depth in [1, 3, 10]:
        model = DecisionTreeRegressor(max_depth=depth)
        model.fit(x_train.reshape(-1, 1), y_train)
        predictions[depth].append(model.predict(x_test.reshape(-1, 1)))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
y_true = true_function(x_test)
for ax, (depth, preds) in zip(axes, predictions.items()):
    preds = np.array(preds)
    mean_pred = preds.mean(axis=0)
    bias_sq = (mean_pred - y_true) ** 2
    variance = preds.var(axis=0)
    for p in preds[::10]:
        ax.plot(x_test, p, 'b-', alpha=0.1, lw=0.8)
    ax.plot(x_test, mean_pred, 'r-', lw=2, label=f'Mean pred')
    ax.plot(x_test, y_true, 'g--', lw=2, label='True function')
    ax.set_title(f'max_depth={depth}
Bias²={bias_sq.mean():.3f}, Var={variance.mean():.3f}')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.suptitle('Bias-Variance Tradeoff: Decision Tree Depth', fontsize=13)
plt.tight_layout()
plt.savefig('bias_variance.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# ROC-AUC and Precision-Recall comparison for imbalanced data
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.model_selection import train_test_split

# Imbalanced dataset
X, y = make_classification(n_samples=10000, n_features=20, n_informative=10,
                            weights=[0.97, 0.03], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc = auc(fpr, tpr)
    ax1.plot(fpr, tpr, lw=2, label=f'{name} (AUC={roc_auc:.3f})')
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    ax2.plot(rec, prec, lw=2, label=f'{name} (AP={ap:.3f})')

ax1.plot([0,1],[0,1],'k--'); ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR')
ax1.set_title('ROC Curve (AUC can look good even with imbalance)'); ax1.legend(); ax1.grid(True,alpha=0.3)
ax2.axhline(y.mean(), color='k', ls='--', label=f'Baseline (AP={y.mean():.3f})')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve (Better for imbalanced data)'); ax2.legend(); ax2.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig('roc_vs_pr.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Class imbalance: {y.mean()*100:.1f}% positive")


## 11. Additional Commonly Asked Questions

**Q_A1. What is the difference between parametric and non-parametric models?**  
**Parametric:** Fixed number of parameters regardless of dataset size (e.g., linear regression, logistic regression, neural networks). Faster inference, more assumptions about data distribution, may underfit complex data.  
**Non-parametric:** Number of parameters grows with data (e.g., KNN, kernel SVM, Gaussian Processes, decision trees without depth limit). More flexible, fewer distribution assumptions, but can be slow at inference and prone to overfitting without regularization.

**Q_A2. What is the difference between discriminative and generative classifiers? Give examples.**  
**Discriminative:** Model P(Y|X) directly. Examples: logistic regression, SVM, CRFs, neural networks. Typically better classification accuracy when enough data exists.  
**Generative:** Model P(X|Y) and P(Y), use Bayes' rule for P(Y|X). Examples: Naive Bayes, GMM, HMMs, VAEs. Can generate new samples, work with unlabeled data (semi-supervised), handle missing features naturally via marginalization.

**Q_A3. Explain the KNN algorithm. What are its computational challenges?**  
K-Nearest Neighbors: classify by majority vote (or average) of the K nearest training points (by Euclidean or other distance). No explicit training — all computation at inference.  
**Challenges:** (1) O(N·d) inference per query — slow for large N; (2) O(N·d) memory — stores all training data; (3) sensitive to irrelevant features (curse of dimensionality); (4) requires feature scaling; (5) choice of K and distance metric is critical. Use KD-Trees or Ball-Trees to speed up for low dimensions (d < 20); for high dimensions, use ANN (FAISS, Annoy).

**Q_A4. What is the difference between overfitting and high variance? Are they the same?**  
Related but not identical. **Overfitting** is a *phenomenon*: model performs well on training data but poorly on new data. **High variance** is the *statistical cause*: model is too sensitive to the specific training set, so predictions change a lot across different training sets. A model can have high variance without severe overfitting (if the data is plentiful) and can technically "overfit" due to high bias in unusual ways. Most overfitting in practice is caused by high variance.

**Q_A5. What is Bayesian optimization for hyperparameter tuning?**  
Builds a probabilistic surrogate model (usually a Gaussian Process) of the objective function (e.g., validation accuracy). Uses an acquisition function (Expected Improvement, UCB) to decide where to evaluate next — balancing exploration and exploitation. Much more efficient than grid/random search: requires far fewer function evaluations to find good hyperparameters. Used in: Hyperopt, Optuna (TPE), Vizier. Best for expensive evaluations.

**Q_A6. What is the difference between bagging and pasting?**  
**Bagging (Bootstrap Aggregating):** Each model trained on a *bootstrap sample* (sampling with replacement → ~63% unique samples). Out-of-bag samples can be used for validation.  
**Pasting:** Sample without replacement — each model sees a unique subset. Less randomness, slightly less regularization. Both reduce variance via averaging. Random Forest uses bagging.

**Q_A7. What is the Gini impurity vs entropy (information gain) — which is better for decision trees?**  
Both measure node impurity. Gini = 1 - Σpᵢ² (faster to compute, no logarithm). Entropy = -Σpᵢ log₂(pᵢ). Practically, they produce very similar trees — the choice rarely matters significantly. Gini is slightly faster. Entropy can be preferred when you want splits closer to equal-frequency distributions. sklearn's default is Gini.

**Q_A8. ⚠️ TRICK: Can a decision tree perfectly fit any training dataset?**  
Yes — a sufficiently deep decision tree (one leaf per sample) achieves zero training error on any dataset (assuming no two identical samples with different labels). This is why decision trees are high-variance models and must be regularized (max_depth, min_samples_leaf, pruning) to generalize.

**Q_A9. ⚠️ TRICK: What is data leakage vs label leakage?**  
**Data leakage:** Using information in training that wouldn't be available at prediction time (future data, target-derived features, wrong CV splits). Leads to optimistically biased evaluation.  
**Label leakage (target leakage):** A subset — using features that are *caused by* or *encode* the target. Example: using a patient's medication as a feature to predict a disease, when the medication was prescribed *because of* the disease. Even more insidious — the feature perfectly predicts the target, but only because of the causal direction.

**Q_A10. What are the assumptions of linear regression and how do you check them?**  
1. **Linearity:** Residuals vs fitted plot (should be random, no pattern)  
2. **Independence:** DW test for autocorrelation (important for time series)  
3. **Homoscedasticity (constant variance):** Scale-location plot (residuals should have constant spread)  
4. **Normality of residuals:** Q-Q plot (residuals should follow a straight line)  
5. **No perfect multicollinearity:** VIF < 5-10 for all features  
Violations don't always render the model useless but affect standard errors, confidence intervals, and p-values.
